## Setup

All the paths used by this exercise are set in the cell below.

In [ ]:
#############################################################
# ALL PATHS ARE SET HERE
# If the data or the software moves, this is the ONLY cell you
# need to change. No cell below this one uses a full path.
#############################################################

# where the shared data lives
DATA=/course/data/PCA/human_lowdepth

# reference data for the locus zoom plot
GENETIC_MAP=/course/data/geneticMap/hg38

# where you will do the exercise
WORK_DIR=$HOME/pca_low_depth_selection_human

mkdir -p $WORK_DIR
echo $WORK_DIR > $HOME/.pca_selection_workdir
echo $GENETIC_MAP > $HOME/.pca_selection_geneticmap
cd $WORK_DIR

# link the input files into the working folder
cp -sf $DATA/eu1000g.small.beagle.gz .
cp -sf $DATA/eu1000g.sample.Info .

echo --programs that are installed:--
which pcangsd

echo; echo --- files in folder ---
ls

In [ ]:
# the working directory and the reference-data folder were set in the first cell
work_d <- readLines(path.expand("~/.pca_selection_workdir"))[1]
setwd(work_d)
GENETIC_MAP <- readLines(path.expand("~/.pca_selection_geneticmap"))[1]
getwd()


# PC-based selection scan

For very recent selection we can look **within** closely related individuals,
using the principal components themselves. A site whose allele frequency changes
along a PC more than drift would allow is a candidate for selection.

The data are low-depth European samples from the 1000 Genomes project. This
exercise continues from [PCA for low depth sequencing](pca_low_depth_human.ipynb).

In [ ]:
# the files were linked into the folder in the setup cell
ls eu1000g*


### Explore the input data. 

Take a quick look at the sample data.

First try to get an overview of the dataset by looking at the information file and making a summary using the following code:
 

In [ ]:
# View first lines of sample info file
echo --- First lines in sample info file
head eu1000g.sample.Info

echo --- Count the number of samples from each population
cut -f 2 -d " " eu1000g.sample.Info | sed 1d| sort | uniq -c 

**Question**
 - How many samples are there from each country?

- How many samples from each country?

Now let's have a look at the genotype likelihood (GL) file that you have created with ANGSD. It is a "beagle format" file called all.beagle.gz - and will be the input file to PCAngsd. The first line in this file is a header line and after that it contains a line for each locus with GLs. By using the unix command wc we can count the number of lines in the file:



In [ ]:
gunzip -c eu1000g.small.beagle.gz | wc -l
 

**Question**
 - Use the line count to work out how many loci the file has genotype likelihoods for.

- Use this to find out how many loci there are GLs for in the data set?



Next, to get an idea of what the GL file contains try (from the command line) to print the first 9 columns of the first 7 lines of the file:



In [ ]:
zcat eu1000g.small.beagle.gz | head -n 7 | cut -f1-9 | column -t

## Ignore the "Broken pipe"

**Questions**
 - The first three columns are the marker name and the two alleles. What do the remaining columns hold, and how many are there per individual?
 - The three numbers for one individual at one site sum to 1. What are they?

In general, the first three columns of a beagle file contain marker name and the two alleles, allele1 and allele2, present in the locus (in beagle A=0, C=1, G=2, T=3).

All following columns contain genotype likelihoods (three columns for each individual: first GL for homozygote for allele1, then GL for heterozygote and then GL for homozygote for allele2). Note that the GL values sum to one per site for each individuals. This is just a normalization of the genotype likelihoods in order to avoid underflow problems in the beagle software it does not mean that they are genotype probabilities.

 - Based on this, what is the most likely genotype of Ind0 in the first locus and the locus six?

### PCAngsd and selection

Run PCangsd with to estimate the covariance matrix while jointly estimating the individuals allele frequencies.



In [ ]:
pcangsd -b eu1000g.small.beagle.gz -o EUsmall -t 5

**Question**
 - This run has no `--selection`. What does PCAngsd produce here, and what will it be used for?

This takes around 2 min to run. The program estimates the covariance matrix that can then be used for PCA. Look at the output from the program.

 - The algorithm might only need a low number of PCs to estimate the allele freuqencies. How many significant PCs (see MAP test in output)?

Now plot the results in R:


In [ ]:
 ## R
 cov <- as.matrix(read.table("EUsmall.cov"))

 e<-eigen(cov)
 ID<-read.table("eu1000g.sample.Info",head=T,stringsAsFactors=T)
 plot(e$vectors[,1:2],col=ID$POP,xlab="PC1",ylab="PC2")

 legend("topleft",fill=1:4,levels(ID$POP))


**Questions**
 - Does the plot look like you expected? Which populations are close to each other and which are distant?
 - Which PC separates the populations best?

 - Does the plot look like you expected? Which populations are close and distant to each other?

Since the European individuals in 1000G are not simple homogeneous disjoint populations it is hard to use PBS/FST or similar statistics to infer selection based on populating differences ( you will learn about these later). However, PCA offers a good description of the differences between individuals without having the define disjoint groups.

Let's try to infer selection along the genome based on the PCA



In [ ]:
pcangsd -b eu1000g.small.beagle.gz -o EUsmall --selection \
    --sites-save --maf 0 -t 5

**Question**
 - The selection scan looks for sites whose allele frequencies vary along a principal component. Why does that make sense as a test for recent selection?


The analysis takes about two minutes. We also need to keep track of whether a SNP is used in the analysis or not, which can be done based on the output. Create a file with the SNP location info that you will need to plot the results (the third column indicate if the site is used=1 or not =0):



In [ ]:
# Create file with position and chromosome
paste <(zcat eu1000g.small.beagle.gz| cut -f 1 | sed 's/\_/\t/g' | sed 1d ) \
    EUsmall.sites > EUsmall.sites.info

echo -- first lines of the created file
head  EUsmall.sites.info 

**Question**
 - Why do we need to keep track of which sites were actually used, rather than assuming every line of the beagle file was?

Next, plot the results of the selection scan



In [ ]:
#read function for plotting
source("https://raw.githubusercontent.com/aalbrechtsen/Rfun/refs/heads/master/online.R")

# read in pvalues from seleciton scan
s <- scan("EUsmall.selection")

# convert test statistic to p-value
pval<-pchisq(s,1,lower=FALSE)

## make QQ plot to QC the test statistics
qqPlot(pval)

### Run the cell below to take the quiz

In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/pca/quiz/pca_selection_qq.json")


**Questions**
 - In a QQ plot, what would it mean if the points followed the diagonal exactly?
 - They do not. Is that selection, or could something else produce it?

The above is a QQ plot of the p-values from the selection scan. If the test statistics is good them most point will follow the red line which only a few (<1%) will deviate.

 - Did the test perform well?
 
 Finally, let's plot the results of the scan along the genome:  

In [ ]:
## read positions (hg38)
data<-read.delim("EUsmall.sites.info",
                 colC=c("factor","integer","integer"),head=F)
names(data)<-c("chr","pos","keep")
data <- subset(data,keep==1)
data$pval <- pval 

## make manhatten plot
options(repr.plot.width = 10, repr.plot.height = 6)
manPlot(data$pval,chr=as.integer(data$chr))

**Question**
 - Which chromosome carries the strongest signal?

Let's zoom in 

In [ ]:
# select sites to plot, 0.5Mb on either side of SNP
leadSNPposition <- data$pos[which.max(s)]

region <- subset(data,chr=="chr2" & pos >  leadSNPposition - 5e5 & 
                 pos < leadSNPposition + 5e5)

#plot
locusZoomNoLD(region$pval,chr=2,pos=region$pos,main="LocusZoom",
    geneticMap=file.path(GENETIC_MAP,"genetic_map_GRCh38_chr"), 
    refGenes=file.path(GENETIC_MAP,"refGeneHG38.gz"),
    w=which(region$pos==leadSNPposition))

**Questions**
 - What do you think is the relevant gene in that region?
 - Does the signal look like a single site or a whole region, and what would you expect under selection?

### Run the cell below to take the quiz

In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/pca/quiz/pca_selection_hit.json")


See if you can make sense of the top hit. What do you think is the relevant gene in  that locus